# 11장. Bag of Words와 문서-단어 행렬

| 항목 | 내용 |
|---|---|
| 예상 시간 | 3시간 |
| 선수 장 | 4장, 10장 |
| 이 장의 결과물 | 어휘 사전, 카운트 행렬 함수, `min_df` 결정 근거 |
| 과제 연결 | "TF 계산: 문서 내 단어 빈도를 행렬로 표현한다" |

**학습 목표**
- BoW가 문서를 어떻게 숫자로 바꾸는지, 무엇을 버리는지 설명할 수 있다.
- 어휘 사전을 정렬해서 만들고, 그 이유를 sklearn과 연결해 말할 수 있다.
- 카운트 행렬을 세 가지 방법으로 만들고 속도를 비교할 수 있다.
- 실제 문서-단어 행렬이 얼마나 희소한지 측정할 수 있다.
- `min_df`, `max_df`로 어휘 크기와 메모리를 조절하고, 값을 고른 근거를 쓸 수 있다.

> 이 장부터는 Part 3에서 만든 **데이터 로딩 + 전처리 코드**를 첫 셀에 그대로 넣고 시작합니다. 영어는 4,646건, 한국어는 7,000건의 토큰 리스트가 준비됩니다(전처리 후 토큰이 너무 적은 문서는 뺐습니다).

In [1]:
# ── 7장에서 만든 데이터 로딩 코드 (src/data.py로 옮길 내용) ──
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
RANDOM_STATE = 42
EN_CATEGORIES = ["comp.graphics", "comp.windows.x", "rec.sport.baseball", "rec.sport.hockey", "sci.space"]
YNAT_URL = ("https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/"
            "klue_benchmark/ynat-v1.1/ynat-v1.1_train.json")
YNAT_PATH = PROJECT / "data" / "raw" / "ynat-v1.1_train.json"

_EMAIL = re.compile(r"\S+@\S+\.\S+")
_URL = re.compile(r"(https?://|www\.)\S+")
_PHONE = re.compile(r"\b\d{2,4}[-.\s]\d{3,4}[-.\s]\d{4}\b")


def mask_pii(text: str) -> str:
    text = _EMAIL.sub(" <email> ", text)
    text = _URL.sub(" <url> ", text)
    return _PHONE.sub(" <phone> ", text)


def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """개인정보 가리기 → 빈 문서 제거 → 레이블 충돌 중복·중복 제거 → 인덱스 재정렬."""
    df = df.assign(text=df["text"].map(mask_pii).str.strip())
    df = df[df["text"] != ""]
    norm = df["text"].str.split().str.join(" ")
    conflict = df.groupby(norm)["label"].transform("nunique") > 1
    df, norm = df[~conflict], norm[~conflict]
    return df[~norm.duplicated(keep="first")].reset_index(drop=True)


def load_english() -> pd.DataFrame:
    bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                               remove=("headers", "footers", "quotes"),
                               shuffle=True, random_state=RANDOM_STATE)
    labels = [bunch.target_names[t] for t in bunch.target]
    return finalize(pd.DataFrame({"text": bunch.data, "label": labels}))


def load_korean(per_class: int = 1000) -> pd.DataFrame:
    if not YNAT_PATH.exists():
        YNAT_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(YNAT_URL, YNAT_PATH)
    raw = pd.read_json(YNAT_PATH).rename(columns={"title": "text"})[["guid", "text", "label"]]
    sampled = (raw.groupby("label", group_keys=False)
                  .sample(n=per_class, random_state=RANDOM_STATE)
                  .sample(frac=1.0, random_state=RANDOM_STATE))
    return finalize(sampled.reset_index(drop=True))

# ── 8~10장에서 만든 전처리 파이프라인 (src/preprocess.py로 옮길 내용) ──
from konlpy.tag import Okt
from nltk.corpus import stopwords

PLACEHOLDER = re.compile(r"<(?:email|url|phone)>")
EN_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")
KO_CLEAN = re.compile(r"[^0-9a-zA-Z가-힣一-鿿\s]")
HANJA_ONLY = re.compile(r"^[一-鿿]+$")
KO_KEEP_POS = {"Noun", "Verb", "Adjective", "Alpha"}
EN_STOP = frozenset(stopwords.words("english"))
DOMAIN_STOPWORDS_EN = frozenset({
    "would", "could", "might", "may", "also", "like", "get", "got", "go", "going", "know", "think",
    "see", "say", "make", "want", "look", "one", "well", "much", "many", "even", "way", "something",
    "anyone", "since", "still", "really", "right", "another", "cant",
})
STOPWORDS_KO = frozenset({
    "하다", "되다", "있다", "없다", "이다", "않다",
    "등", "것", "수", "중", "및", "때", "후", "내", "듯", "제", "개", "의", "로", "종합",
})
_okt = Okt()


def clean_en(text: str) -> str:
    text = PLACEHOLDER.sub(" ", text.lower()).replace("’", "'")
    return " ".join(EN_TOKEN.findall(text))


def clean_ko(text: str) -> str:
    return " ".join(KO_CLEAN.sub(" ", PLACEHOLDER.sub(" ", text)).split())


class EnglishPreprocessor:
    def __init__(self, use_domain_stopwords: bool = True, min_len: int = 2, max_len: int = 20):
        self.stopwords = EN_STOP | (DOMAIN_STOPWORDS_EN if use_domain_stopwords else frozenset())
        self.min_len, self.max_len = min_len, max_len

    def __call__(self, text: str) -> list[str]:
        tokens = []
        for tok in clean_en(text).split():
            if tok in self.stopwords:
                continue
            tok = tok.removesuffix("'s").replace("'", "")
            if self.min_len <= len(tok) <= self.max_len and tok not in self.stopwords:
                tokens.append(tok)
        return tokens


class KoreanPreprocessor:
    def __init__(self, tokenizer: str = "okt", use_stopwords: bool = True):
        if tokenizer not in ("okt", "whitespace"):
            raise ValueError(f"tokenizer는 'okt' 또는 'whitespace': {tokenizer}")
        self.tokenizer = tokenizer
        self.stopwords = STOPWORDS_KO if use_stopwords else frozenset()

    def tokenize(self, text: str) -> list[str]:
        if self.tokenizer == "whitespace":
            return clean_ko(text).lower().split()
        return [w.lower() for w, p in _okt.pos(clean_ko(text), norm=True, stem=True)
                if p in KO_KEEP_POS or (p == "Foreign" and HANJA_ONLY.match(w))]

    def __call__(self, text: str) -> list[str]:
        return [t for t in self.tokenize(text) if t not in self.stopwords and not t.isdigit()]

# ── 로드 + 전처리를 한 번에 (src/corpus.py로 옮길 내용) ──
MIN_TOKENS = {"en": 5, "ko": 2}   # 전처리 후 토큰이 이보다 적은 문서는 뺀다 (짧은 답글, 주제어 없는 제목)
MIN_DF = {"en": 3, "ko": 2}       # 어휘에 넣을 최소 문서 빈도 (11장)


def load_corpus(lang: str, tokenizer: str = "okt") -> pd.DataFrame:
    """열: text, label, tokens. lang은 "en" 또는 "ko"."""
    df = load_english() if lang == "en" else load_korean()
    prep = EnglishPreprocessor() if lang == "en" else KoreanPreprocessor(tokenizer)
    df["tokens"] = [prep(t) for t in df["text"]]
    df = df[df["tokens"].map(len) >= MIN_TOKENS[lang]].reset_index(drop=True)
    df.attrs["preprocessor"] = prep
    return df

# ── 13장에서 만든 NumPy TF-IDF (src/tfidf.py로 옮길 내용) ──
class NumpyTfidf:
    """TF(raw 또는 log) × smooth IDF, 문서마다 L2 정규화. sklearn TfidfVectorizer와 같은 정의."""

    def __init__(self, tf: str = "raw", smooth_idf: bool = True, norm: str | None = "l2",
                 min_df: int = 1, max_df: float = 1.0):
        if tf not in ("raw", "log"):
            raise ValueError(f"tf는 'raw' 또는 'log': {tf}")
        if norm not in ("l2", None):
            raise ValueError(f"norm은 'l2' 또는 None: {norm}")
        self.tf, self.smooth_idf, self.norm = tf, smooth_idf, norm
        self.min_df, self.max_df = min_df, max_df

    def fit(self, docs: list[list[str]]) -> "NumpyTfidf":
        n = len(docs)
        df: dict[str, int] = {}
        for doc in docs:
            for t in set(doc):
                df[t] = df.get(t, 0) + 1
        max_count = self.max_df * n if isinstance(self.max_df, float) else self.max_df
        terms = sorted(t for t, c in df.items() if self.min_df <= c <= max_count)
        self.vocabulary_ = {t: i for i, t in enumerate(terms)}
        self.terms_ = np.array(terms, dtype=object)
        self.n_docs_ = n
        self.df_ = np.array([df[t] for t in terms], dtype=np.float64)
        if self.smooth_idf:
            self.idf_ = np.log((1.0 + n) / (1.0 + self.df_)) + 1.0
        else:
            self.idf_ = np.log(n / self.df_) + 1.0
        return self

    def count_matrix(self, docs: list[list[str]]) -> np.ndarray:
        rows, cols = [], []
        for i, doc in enumerate(docs):
            for t in doc:
                j = self.vocabulary_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        counts = np.zeros((len(docs), len(self.vocabulary_)), dtype=np.float64)
        np.add.at(counts, (np.asarray(rows, dtype=np.int64), np.asarray(cols, dtype=np.int64)), 1.0)
        return counts

    def transform(self, docs: list[list[str]]) -> np.ndarray:
        x = self.count_matrix(docs)
        if self.tf == "log":
            mask = x > 0
            x[mask] = 1.0 + np.log(x[mask])
        x *= self.idf_                                   # (n, V) * (V,) 브로드캐스팅
        if self.norm == "l2":
            norms = np.linalg.norm(x, axis=1, keepdims=True)
            x /= np.where(norms > 0, norms, 1.0)          # 빈 문서는 0 벡터로 둔다
        return x

    def fit_transform(self, docs: list[list[str]]) -> np.ndarray:
        return self.fit(docs).transform(docs)

# 미니 코퍼스: 모든 장에서 함께 쓰는 문서 5개 (0장 0.5절 참고)
MINI_DOCS = [
    "The rocket launched into space!",
    "NASA launched a new space telescope.",
    "The pitcher threw a fast ball.",
    "Our team won the baseball game.",
    "Astronauts watched the baseball game in space.",
]
MINI_LABELS = ["space", "space", "baseball", "baseball", "space"]

import numpy as np
import pandas as pd

STOP = {"the", "a", "into", "our", "in", "new"}
FILTERED = [[t for t in doc.lower().strip(".!").split() if t not in STOP] for doc in MINI_DOCS]
np.set_printoptions(precision=4, suppress=True, linewidth=120)

import time
from collections import Counter

t0 = time.perf_counter()
en = load_corpus("en")
ko = load_corpus("ko")
print(f"영어 {len(en):,}건, 한국어 {len(ko):,}건 ({time.perf_counter() - t0:.0f}초)")
print("영어 첫 문서 토큰 앞 15개:", en.loc[0, "tokens"][:15])
print("한국어 첫 문서 토큰      :", ko.loc[0, "tokens"])

영어 4,646건, 한국어 7,000건 (9초)
영어 첫 문서 토큰 앞 15개: ['according', 'osf', 'motif', 'style', 'guide', 'use', 'cursor', 'shapes', 'give', 'user', 'visual', 'clue', 'happening', 'expected', 'hourglass']
한국어 첫 문서 토큰      : ['朴', '대통령', '지방', '중앙', '힘', '모으다', '위기', '기회']


## 11.1 Bag of Words: 순서를 버리고 가방에 담기

**BoW(Bag of Words)**는 문서를 "어떤 단어가 몇 번 들어 있는가"로만 표현합니다. 단어를 가방에 쏟아 넣는다고 생각하면 됩니다. 가방 안에서는 **순서가 사라지고 개수만 남습니다.**

In [2]:
a = "dog bites man".split()
b = "man bites dog".split()
print("문장 A:", a, "→", dict(sorted(Counter(a).items())))
print("문장 B:", b, "→", dict(sorted(Counter(b).items())))
print("BoW로는 같은가?", Counter(a) == Counter(b))

문장 A: ['dog', 'bites', 'man'] → {'bites': 1, 'dog': 1, 'man': 1}
문장 B: ['man', 'bites', 'dog'] → {'bites': 1, 'dog': 1, 'man': 1}
BoW로는 같은가? True


뜻이 정반대인 두 문장이 BoW로는 **완전히 같습니다.** 이것이 BoW의 가장 큰 한계이고(21장), 그 대가로 얻는 것은 **단순함**입니다. 문서가 아무리 길어도 "어휘 크기만큼의 숫자 배열" 하나가 되므로, 3–4장의 벡터 계산을 그대로 쓸 수 있습니다.

| BoW가 남기는 것 | BoW가 버리는 것 |
|---|---|
| 어떤 단어가 있는가 | 단어의 순서 (어순) |
| 몇 번 나왔는가 | 문맥 (같은 단어의 다른 뜻) |
| | 단어끼리의 관계 (동의어) |

## 11.2 어휘 사전 만들기

어휘 사전은 **단어 → 열 번호** 딕셔너리입니다. 모든 문서가 같은 사전을 써야 같은 열이 같은 단어를 뜻합니다.

In [3]:
vocab = sorted({t for doc in FILTERED for t in doc})
word_to_index = {w: i for i, w in enumerate(vocab)}
print(word_to_index)

{'astronauts': 0, 'ball': 1, 'baseball': 2, 'fast': 3, 'game': 4, 'launched': 5, 'nasa': 6, 'pitcher': 7, 'rocket': 8, 'space': 9, 'team': 10, 'telescope': 11, 'threw': 12, 'watched': 13, 'won': 14}


### 왜 정렬하나
`set`은 순서가 정해져 있지 않아서, 실행할 때마다(또는 문서 순서가 바뀌면) 열 번호가 달라질 수 있습니다. **정렬하면 같은 어휘에서 항상 같은 열 번호**가 나옵니다.

더 중요한 이유는 14장 검증입니다. sklearn의 `CountVectorizer`/`TfidfVectorizer`도 어휘를 **정렬**해서 열 번호를 붙입니다. 내 구현도 정렬해 두면 두 행렬의 열이 처음부터 같은 단어를 가리킵니다.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer


def identity(doc):          # 이미 토큰 리스트이므로 그대로 돌려준다 (14장에서 자세히)
    return doc


cv = CountVectorizer(analyzer=identity, lowercase=False).fit(FILTERED)
print("sklearn 열 순서:", list(cv.get_feature_names_out()))
print("정렬한 어휘와 같은가:", list(cv.get_feature_names_out()) == vocab)

sklearn 열 순서: ['astronauts', 'ball', 'baseball', 'fast', 'game', 'launched', 'nasa', 'pitcher', 'rocket', 'space', 'team', 'telescope', 'threw', 'watched', 'won']
정렬한 어휘와 같은가: True


파이썬의 문자열 정렬은 **유니코드 코드 순서**입니다. 한국어 어휘에서는 숫자·영문 → 한자(`一`=U+4E00부터) → 한글(`가`=U+AC00부터) 순서가 됩니다. sklearn도 같은 `sorted()`를 쓰므로 순서가 일치합니다.

In [5]:
ko_vocab = sorted({t for doc in ko["tokens"] for t in doc})
print("앞 8개:", ko_vocab[:8])
hanja_start = next(i for i, w in enumerate(ko_vocab) if "一" <= w[0] <= "鿿")
hangul_start = next(i for i, w in enumerate(ko_vocab) if "가" <= w[0] <= "힣")
print("한자 시작:", ko_vocab[hanja_start:hanja_start + 5])
print("한글 시작:", ko_vocab[hangul_start:hangul_start + 5])

앞 8개: ['a', 'abs', 'afc', 'ag', 'ah', 'ai', 'aia', 'al']
한자 시작: ['丁', '中', '中企', '亞', '人']
한글 시작: ['가격', '가결', '가계', '가계부채', '가곡']


## 11.3 카운트 행렬 만들기: 세 가지 방법

문서 $i$에 단어 $j$가 나온 횟수를 $(i, j)$ 칸에 넣은 행렬이 **카운트 행렬**(문서-단어 행렬)입니다. 4장에서 미니 코퍼스로 해 본 것을 실제 데이터에 적용하며 세 가지 방법을 비교합니다. 영어 앞 1,000건과, 문서 빈도 3 이상인 어휘(11.5절에서 이유를 다룹니다)를 씁니다.

In [6]:
docs = en["tokens"].tolist()[:1000]
en_vec = NumpyTfidf(min_df=3).fit(en["tokens"].tolist())     # 어휘 사전만 빌려 쓴다
vocab_en = en_vec.vocabulary_
n, V = len(docs), len(vocab_en)
print(f"문서 {n:,}건 × 어휘 {V:,}개")


def counts_double_loop(docs):
    X = np.zeros((len(docs), V))
    for i, doc in enumerate(docs):
        for t in doc:
            j = vocab_en.get(t)
            if j is not None:             # 어휘에 없는 단어(OOV)는 버린다
                X[i, j] += 1
    return X


def counts_counter(docs):
    X = np.zeros((len(docs), V))
    for i, doc in enumerate(docs):
        for t, c in Counter(doc).items():  # 같은 단어는 한 번에 넣는다
            j = vocab_en.get(t)
            if j is not None:
                X[i, j] = c
    return X


def counts_add_at(docs):
    rows, cols = [], []
    for i, doc in enumerate(docs):
        for t in doc:
            j = vocab_en.get(t)
            if j is not None:
                rows.append(i)
                cols.append(j)
    X = np.zeros((len(docs), V))
    np.add.at(X, (np.array(rows), np.array(cols)), 1)   # 중복 위치도 정확히 누적 (4장)
    return X


results = {}
for name, f in [("이중 루프", counts_double_loop), ("Counter", counts_counter), ("np.add.at", counts_add_at)]:
    t0 = time.perf_counter()
    results[name] = f(docs)
    print(f"{name:<10} {time.perf_counter() - t0:.3f}초")
print("세 결과가 같은가:", all(np.array_equal(results["이중 루프"], X) for X in results.values()))
del results

문서 1,000건 × 어휘 13,387개
이중 루프      0.032초
Counter    0.022초
np.add.at  0.026초
세 결과가 같은가: True


세 방법의 결과는 같습니다. 속도 차이는 크지 않은데, 어느 방법이든 **토큰을 하나씩 사전에서 찾는 파이썬 루프**가 남아 있기 때문입니다. 차이는 "칸에 값을 넣는 일"을 누가 하느냐입니다.

| 방법 | 칸에 값 넣기 | 특징 |
|---|---|---|
| 이중 루프 | 토큰마다 NumPy 원소 하나씩 수정 | 가장 이해하기 쉬움. NumPy 원소 접근이 느림 |
| Counter | 단어 종류마다 한 번 | 같은 단어가 많이 반복되는 긴 문서에서 유리 |
| `np.add.at` | 위치 목록을 모아 **한 번에** | 계산을 NumPy에 맡김. 13장 구현에서 쓰는 방법 |

> 실행할 때마다 시간은 조금씩 달라집니다. 순위보다 "세 방법이 같은 결과를 낸다"는 것과 각 방법의 아이디어를 기억하세요.

## 11.4 희소성 측정

문서 하나에는 어휘 중 아주 일부만 나옵니다. 문서별로 **0이 아닌 칸 수**(서로 다른 단어 수)를 세면, 밀집 행렬을 만들지 않고도 희소성을 잴 수 있습니다.

In [7]:
def sparsity_report(token_lists, min_df):
    vec = NumpyTfidf(min_df=min_df).fit(token_lists)
    nnz = np.array([len({t for t in doc if t in vec.vocabulary_}) for doc in token_lists])
    n, V = len(token_lists), len(vec.vocabulary_)
    return {"문서 수": n, "어휘 수": V,
            "문서당 0이 아닌 칸 (평균)": round(nnz.mean(), 1),
            "문서당 0이 아닌 칸 (최대)": int(nnz.max()),
            "0인 칸 비율": f"{1 - nnz.sum() / (n * V):.2%}",
            "밀집 행렬 (MB)": round(n * V * 8 / 1024**2),
            "희소 행렬 추정 (MB)": round(nnz.sum() * 12 / 1024**2, 1)}


pd.DataFrame({"영어 (min_df=3)": sparsity_report(en["tokens"].tolist(), 3),
              "한국어 (min_df=2)": sparsity_report(ko["tokens"].tolist(), 2)})

,영어 (min_df=3),한국어 (min_df=2)
문서 수,4646,7000
어휘 수,13387,6332
문서당 0이 아닌 칸 (평균),55.9,7.2
문서당 0이 아닌 칸 (최대),1793,14
0인 칸 비율,99.58%,99.89%
밀집 행렬 (MB),475,338
희소 행렬 추정 (MB),3.0,0.6


- 두 데이터 모두 **99% 이상이 0**입니다. 한국어는 제목이 짧아서 문서당 단어가 몇 개 되지 않습니다.
- **희소 행렬**(0이 아닌 칸의 값과 위치만 저장, 칸당 약 12바이트)으로 저장하면 메모리가 백 분의 일 이하로 줄어듭니다. sklearn이 희소 행렬을 돌려주는 이유입니다.
- 과제는 NumPy만으로 구현하라고 하므로 이 자료에서는 **밀집 행렬**을 씁니다. 그래서 어휘 크기를 조절하는 것이 중요합니다(다음 절).

## 11.5 어휘 크기 조절: min_df와 max_df

| 설정 | 뜻 | 지우는 것 |
|---|---|---|
| `min_df=k` (정수) | 문서 **k개 이상**에 나온 단어만 남긴다 | 오타, 고유명사 잡음, 한 번 쓰고 마는 단어 |
| `max_df=0.5` (실수) | 전체 문서의 **50% 이하**에 나온 단어만 남긴다 | 너무 흔한 단어 (자동 불용어) |
| `max_df=100` (정수) | 문서 **100개 이하**에 나온 단어만 | 같음, 개수 기준 |

sklearn과 같은 규칙입니다. **실수는 비율, 정수는 문서 수**로 해석합니다.

In [8]:
rows = []
for lang, df_ in [("en", en), ("ko", ko)]:
    token_lists = df_["tokens"].tolist()
    for m in [1, 2, 3, 5, 10]:
        V = len(NumpyTfidf(min_df=m).fit(token_lists).vocabulary_)
        rows.append({"데이터": lang, "min_df": m, "어휘 수": V, "밀집 행렬 (MB)": round(len(token_lists) * V * 8 / 1024**2)})
table = pd.DataFrame(rows).pivot(index="min_df", columns="데이터", values=["어휘 수", "밀집 행렬 (MB)"])
table

어휘 수        밀집 행렬 (MB)     
데이터        en     ko         en   ko
min_df                              
1       35004  11383       1241  608
2       18628   6332        660  338
3       13387   4304        475  230
5        9009   2635        319  141
10       5307   1268        188   68

- `min_df=1`(아무것도 거르지 않음)과 `min_df=2`의 차이가 가장 큽니다. **한 문서에만 나온 단어가 어휘의 거의 절반**이라는 뜻입니다. 이런 단어는 다른 문서와 공유되지 않으므로 검색 유사도에도, 분류 학습에도 거의 쓸모가 없습니다.
- 영어는 `min_df=2`로도 밀집 행렬이 수백 MB입니다. Train/Test 분할, 검증용 sklearn 행렬, 오차 행렬까지 동시에 메모리에 올라가면 몇 GB가 됩니다. 그래서 영어는 **`min_df=3`**을 씁니다.
- 한국어는 제목이 짧아 단어가 드물게 나오므로, 너무 많이 거르면 문서에 남는 단어가 없어집니다. **`min_df=2`**를 씁니다.

`min_df`를 올리면 문서마다 남는 단어가 줄어듭니다. 단어가 하나도 남지 않는 문서가 생기는지도 확인합니다.

In [9]:
for lang, df_, m in [("en", en, 3), ("ko", ko, 2), ("ko", ko, 5)]:
    vec = NumpyTfidf(min_df=m).fit(df_["tokens"].tolist())
    left = df_["tokens"].map(lambda d: sum(t in vec.vocabulary_ for t in d))
    print(f"{lang} min_df={m}: 어휘 밖 단어만 남은 문서 {int((left == 0).sum())}건, 남은 토큰 수 중앙값 {left.median():.0f}")

en min_df=3: 어휘 밖 단어만 남은 문서 0건, 남은 토큰 수 중앙값 36
ko min_df=2: 어휘 밖 단어만 남은 문서 1건, 남은 토큰 수 중앙값 7
ko min_df=5: 어휘 밖 단어만 남은 문서 16건, 남은 토큰 수 중앙값 6


한국어는 `min_df=2`에서도 단어가 모두 빠지는 문서가 생기고, `min_df=5`로 올리면 그런 문서가 크게 늘어납니다. 이런 문서는 0 벡터가 되어 검색에도 분류에도 쓸 정보가 없습니다. 짧은 제목 데이터에서 `min_df`를 낮게 두는 이유입니다.

### max_df: 자동 불용어
`max_df`를 낮추면 너무 흔한 단어가 빠집니다. 10장에서 직접 고른 불용어와 비교해 봅니다.

In [10]:
for lang, df_ in [("en", en), ("ko", ko)]:
    token_lists = df_["tokens"].tolist()
    full = NumpyTfidf(min_df=1).fit(token_lists)
    cut = NumpyTfidf(min_df=1, max_df=0.05).fit(token_lists)
    removed = sorted(set(full.vocabulary_) - set(cut.vocabulary_),
                     key=lambda t: -full.df_[full.vocabulary_[t]])
    print(f"{lang}: max_df=0.05로 빠지는 단어 {len(removed)}개 →", removed[:15])

en: max_df=0.05로 빠지는 단어 105개 → ['time', 'use', 'good', 'first', 'thanks', 'last', 'two', 'game', 'year', 'new', 'please', 'team', 'need', 'using', 'people']
ko: max_df=0.05로 빠지는 단어 0개 → []


영어에서 문서의 5% 넘게 나오는 단어에는 `game`, `team` 같은 **주제어도 섞여 있습니다.** 한국어는 제목이 짧고 불용어를 이미 지워서, 5% 넘게 나오는 단어가 하나도 없습니다. `max_df`는 단어의 성격을 보지 않고 빈도만으로 자르기 때문입니다. 10장처럼 카테고리 균등도를 보고 직접 고른 불용어가 더 안전하므로, 이 자료에서는 `max_df`를 쓰지 않습니다(기본값 1.0).

> **주의:** `min_df`, `max_df`는 어휘를 **fit한 문서 기준**으로 계산합니다. 분류에서 Train/Test를 나누면 Train 문서만으로 fit해야 합니다(17장, 데이터 누수).

## ✍️ 실습: 미니 코퍼스 카운트 행렬 손으로 쓰기

1. 종이에 미니 코퍼스(`FILTERED`)의 어휘를 알파벳 순으로 쓰고, 5×15 카운트 행렬을 손으로 채우세요.
2. 손으로 쓴 행렬을 `HAND = np.array([...])`로 옮기고, `counts_add_at` 방식으로 만든 행렬과 같은지 확인하세요.

In [11]:
for i, doc in enumerate(FILTERED):
    print(i, doc)

0 ['rocket', 'launched', 'space']
1 ['nasa', 'launched', 'space', 'telescope']
2 ['pitcher', 'threw', 'fast', 'ball']
3 ['team', 'won', 'baseball', 'game']
4 ['astronauts', 'watched', 'baseball', 'game', 'space']


In [12]:
# 여기에 작성하세요

**정답**

In [13]:
# astronauts ball baseball fast game launched nasa pitcher rocket space team telescope threw watched won
HAND = np.array([
    [0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0],   # doc 0: rocket launched space
    [0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0],   # doc 1: nasa launched space telescope
    [0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0],   # doc 2: pitcher threw fast ball
    [0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],   # doc 3: team won baseball game
    [1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0],   # doc 4: astronauts watched baseball game space
])

mini_vec = NumpyTfidf().fit(FILTERED)
code_counts = mini_vec.count_matrix(FILTERED)
print("어휘:", list(mini_vec.vocabulary_))
print("손 계산과 같은가:", np.array_equal(HAND, code_counts))

어휘: ['astronauts', 'ball', 'baseball', 'fast', 'game', 'launched', 'nasa', 'pitcher', 'rocket', 'space', 'team', 'telescope', 'threw', 'watched', 'won']
손 계산과 같은가: True


## 🧐 퀴즈

**Q1.** "dog bites man"과 "man bites dog"의 BoW 벡터가 같은 이유와, 이것이 보여 주는 BoW의 한계는?
<details><summary>▶ 정답 보기</summary>

BoW는 단어의 개수만 세고 순서를 버리기 때문입니다. 어순이 바뀌어 뜻이 정반대가 되어도 같은 벡터가 됩니다(어순 무시, 21장).
</details>

**Q2.** 어휘 사전을 정렬해서 만드는 이유 두 가지는?
<details><summary>▶ 정답 보기</summary>

① 같은 어휘에서 항상 같은 열 번호가 나와 결과를 재현할 수 있고, ② sklearn도 어휘를 정렬하므로 14장에서 두 행렬을 열 단위로 바로 비교할 수 있습니다.
</details>

**Q3.** `min_df=1`에서 `min_df=2`로 바꾸면 어휘가 크게 줄어듭니다. 무엇이 빠진 것이고, 왜 빼도 괜찮은가요?
<details><summary>▶ 정답 보기</summary>

한 문서에만 나온 단어가 빠집니다. 이런 단어는 다른 문서와 공유되지 않아 검색 유사도에 기여하지 못하고, 분류 모델도 한 번 본 단어로는 일반화할 수 없습니다. 오타나 고유한 잡음도 많습니다.
</details>

**Q4.** `max_df=0.5`와 `max_df=50`은 어떻게 다른가요?
<details><summary>▶ 정답 보기</summary>

실수 0.5는 비율이라 "전체 문서의 50% 이하에 나온 단어", 정수 50은 개수라 "문서 50개 이하에 나온 단어"만 남깁니다.
</details>

**Q5.** 영어에 `min_df=3`, 한국어에 `min_df=2`를 고른 근거는?
<details><summary>▶ 정답 보기</summary>

영어는 본문이 길어 어휘가 많아서, `min_df=2`로도 밀집 행렬이 수백 MB라 검증·분할 과정에서 메모리가 부족해질 수 있습니다. 3으로 올리면 어휘와 메모리가 크게 줄어듭니다. 한국어는 제목이 짧아 단어가 드물게 나오므로, 많이 거르면 문서에 남는 단어가 없어져 2로 둡니다.
</details>

## ✅ 체크포인트
- [ ] BoW가 남기는 정보와 버리는 정보를 말할 수 있다.
- [ ] 정렬된 어휘 사전과 카운트 행렬을 만들 수 있다.
- [ ] 문서-단어 행렬의 희소성과 밀집 행렬 메모리를 계산할 수 있다.
- [ ] `min_df`, `max_df` 값을 정하고 근거를 쓸 수 있다.

## 🔗 과제 연결과 다음 장
- 카운트 행렬은 과제 요구사항 "TF 계산: 문서 내 단어 빈도를 행렬로 표현한다"에 해당합니다. 13장 `count_matrix` 메서드가 됩니다.
- `min_df` 결정 근거(어휘 수, 메모리 표)는 README의 전처리·설계 근거에 씁니다.
- **12장. TF-IDF 수식**에서 이 카운트 행렬에 가중치를 매기는 방법을 손으로 계산합니다.